In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from global_model_periodic_energy import LearnedSimulator_periodic
import torch.nn as nn
import torch.optim as optim
from scipy.spatial import Voronoi

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')
normalization_stats = {
    "velocity": {"mean": torch.tensor([0.0, 0.0]).to(device), "std": torch.tensor([1e-3, 1e-3]).to(device)},
    "acceleration": {"mean": torch.tensor([0.0, 0.0]).to(device), "std": torch.tensor([1e0, 1e0]).to(device)}
}

In [3]:
checkpoint = torch.load("/home/jeanlienhard/Documents/Cell_GNN/GNN for energy/GNN_for _energy_target/train/test/model_300.pth")
new_state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint.items()}
model = LearnedSimulator_periodic(num_dimensions=2, normalization_stats=normalization_stats, device=device,n_cells = 20)
model.load_state_dict(new_state_dict)
model.to(device)

LearnedSimulator_periodic(
  (graph_network): EnergyGNN(
    (edge_to_node): edgeToNode()
    (gnn_layer1): NodeGNN()
    (gnn_layer2): NodeGNN()
    (gnn_layer3): NodeGNN()
    (gnn_layer4): NodeGNN()
    (gnn_layer5): NodeGNN()
    (regressor): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=1, bias=True)
    )
  )
)

In [4]:
n_cells = 20
n_steps = 100   # nombre d'étapes simulées
dt = 0.1  # pas de temps
max_iter_per_step = 100  # iterations internes de LBFGS par step
lr = 1


In [5]:
df = pd.read_csv("/home/jeanlienhard/Documents/Cell_GNN/Data/raw_data/positions_21.csv")
x0 = df[df["step"] == 0].iloc[:n_cells][['x', 'y']].values.astype(np.float32)

# positions initiales comme variable optimisable
positions = torch.tensor(x0, dtype=torch.float32, device=device, requires_grad=True)
prev_prev_positions = positions.clone().detach()
prev_positions = positions.clone().detach()
# pour stocker la trajectoire
trajectory = [positions.detach().cpu().numpy()]

# vitesse initiale nulle
prev_positions = positions.clone().detach()

In [6]:
def make_periodic_copies(x_centers):
    x_offset, y_offset = 2.0, 2.0
    copies = [x_centers]
    for gx in range(-2,3):
        for gy in range(-2,3):
            if gx != 0 or gy != 0:
                shift = torch.tensor([gx*x_offset, gy*y_offset], device=x_centers.device)
                copies.append(x_centers + shift)
    x_full = torch.cat(copies, dim=0)
    return x_full.unsqueeze(1)


In [7]:
def generate_area_and_perimeter(n_cells):
            np.random.seed(123)
            # Générer des aires aléatoires positives    
            min_area = 0.1
            while True:
                raw = np.random.rand(n_cells).astype(np.float32)#np.array([1.0]*n_cells).astype(np.float32)#
                raw /= raw.sum()
                areas = raw * 4.0
                if np.all(areas >= min_area):
                    break 
            a = np.sqrt((2 * areas) / (3 * np.sqrt(3)))
            perimeters = np.array(6 * a)#
            return torch.from_numpy(areas).to(device), torch.from_numpy(perimeters.astype(np.float32)).to(device)

In [8]:
optimizer = torch.optim.LBFGS([positions], lr=lr, max_iter=max_iter_per_step, history_size=10, line_search_fn="strong_wolfe")
areas,perimeters = generate_area_and_perimeter(n_cells)
print(areas,perimeters)
for step in range(n_steps):
    def closure():
        optimizer.zero_grad()
        x_centers = positions
        
        x_full = make_periodic_copies(x_centers)
        E_cell = model(x_full, n_cells,areas,perimeters)
        E_potential = E_cell.sum()
        
        acceleration = (positions - 2 * prev_positions + prev_prev_positions) / (dt ** 2)
        accel_penalty = 0.5*(acceleration ** 2).sum()
        #print(accel_penalty/E_potential)
        total_loss = 1e-4*E_potential + 1.83295*1e-5*accel_penalty

        total_loss.backward()

        for p in optimizer.param_groups[0]['params']:
            if p.grad is not None and not p.grad.is_contiguous():
                p.grad = p.grad.contiguous()
        return total_loss
    
    optimizer.step(closure)
    
    trajectory.append(positions.detach().cpu().numpy())
    prev_prev_positions = prev_positions.clone().detach()
    prev_positions = positions.clone().detach()



tensor([0.1352, 0.1069, 0.3314, 0.2104, 0.1979, 0.1146, 0.2658, 0.3365, 0.2016,
        0.1784, 0.1235, 0.2140, 0.2074, 0.1648, 0.1816, 0.1338, 0.2084, 0.2387,
        0.3114, 0.1379], device='cuda:0') tensor([1.3685, 1.2170, 2.1428, 1.7073, 1.6558, 1.2603, 1.9192, 2.1595, 1.6714,
        1.5722, 1.3084, 1.7219, 1.6951, 1.5111, 1.5862, 1.3615, 1.6991, 1.8187,
        2.0771, 1.3823], device='cuda:0')


In [9]:
total_pos = []
step = 0
print(len(trajectory))
for traj in trajectory:
    x = torch.tensor(traj, requires_grad=True, dtype=torch.float32, device=device)
    x_centers = x.view(20, 2)
    full = make_periodic_copies(x_centers)
    for site_index in range(full.shape[0]):
            total_pos.append([step, site_index, full[site_index][0][0].item(), full[site_index][0][1].item()])
    step += 1
trajectories_df = pd.DataFrame(total_pos, columns=['step', 'site_index', 'x', 'y'])
trajectories_df.to_csv("trajectories_20.csv", index=False)

101


In [10]:
x0_tensor = torch.tensor(x0.reshape(20,2), device=device, dtype=torch.float32)
x0_full = make_periodic_copies(x0_tensor).squeeze(1).detach().cpu().numpy()
df_initial = pd.DataFrame(x0_full, columns=['x','y'])
df_initial.to_csv("positions_initiales_periodicite_20.csv", index=False)

In [11]:
final_x_full = make_periodic_copies(positions)
final_x_full_np = final_x_full.squeeze(1).detach().cpu().numpy() 
E_cell = model(final_x_full, n_cells,areas,perimeters)
print(E_cell,sum(E_cell))
df = pd.DataFrame(final_x_full_np, columns=['x', 'y'])
df.to_csv("positions_optimisees_periodicite_20.csv", index=False)

tensor([0.3444, 0.3037, 0.5948, 0.7954, 0.2499, 0.5022, 0.4214, 1.3540, 0.3185,
        0.7763, 0.8601, 0.7633, 0.5260, 0.1275, 1.1036, 1.0161, 0.4259, 1.0698,
        0.7806, 0.3858], device='cuda:0', grad_fn=<SqueezeBackward1>) tensor(12.7193, device='cuda:0', grad_fn=<AddBackward0>)


In [12]:
def compute_polygon_area_and_perimeter(polygon):
    polygon = np.array(polygon)
    x = polygon[:, 0]
    y = polygon[:, 1]
    area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    perimeter = np.sum(np.linalg.norm(np.roll(polygon, -1, axis=0) - polygon, axis=1))
    return area, perimeter

def vornoi_area_and_perimeter(vor,target_indices):
    areas = []
    perimeters = []

    for idx in target_indices:
        region_index = vor.point_region[idx]
        region = vor.regions[region_index]
        if -1 in region or len(region) == 0:
            areas.append(1e-10)
            perimeters.append(1e-10)
            continue
        polygon = [vor.vertices[i] for i in region]
        area, perimeter = compute_polygon_area_and_perimeter(polygon)
        areas.append(area)
        perimeters.append(perimeter)          
    return areas,perimeters

In [13]:
def voronoi_loss(output,target_indices,target_areas,target_perimeters,masse = 0.1, dt = 0.05):
    areas = []
    perimeters = []
    vor = Voronoi(output.cpu().detach().numpy())
    area,perimeter = vornoi_area_and_perimeter(vor,target_indices)
    areas.append(area)
    perimeters.append(perimeter)
    areas_tensor = torch.tensor(np.array(areas), dtype=torch.float32).to(device)
    perimeters_tensor = torch.tensor(np.array(perimeters),dtype=torch.float32).to(device)
    # areas_tensor = torch.stack(areas)
    # perimeters_tensor = torch.stack(perimeters)
    physics_loss = 0.02*(target_areas - areas_tensor)**2 + 0.005*(target_perimeters-perimeters_tensor)**2#+1e-5*(target_areas/(areas_tensor))**2
    # kinetic_loss = torch.sum((0.5*masse*dt**2)*(accelerations/(dt**2))**2,dim=-1)
    # print(physics_loss,kinetic_loss)
    return (physics_loss).squeeze(0)#+kinetic_loss

In [14]:
target_indices = np.arange(20)
print(1e4*voronoi_loss(final_x_full.view(500,2),target_indices,areas,perimeters))

tensor([0.0816, 0.1599, 1.3797, 0.3320, 0.2045, 0.1381, 0.0269, 0.5169, 0.0601,
        0.1937, 1.8320, 0.3557, 1.4554, 0.0215, 0.0484, 4.3834, 0.0369, 0.7404,
        0.3673, 0.2085], device='cuda:0')
